In [12]:
"""
ADD THIS TO YOUR JUPYTER NOTEBOOK
Paste this at the top of your existing notebook cell (or as a new cell)
It sends every reading to the dashboard automatically
"""

import time
import json
import requests
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
POLL_INTERVAL  = 3.0
LOG_TO_FILE    = True
LOG_FILE       = "wokwi_log.json"
BACKEND_URL    = "http://localhost:8080/api/vitals"   # Your backend
PATIENT_ID     = "P001"                               # This hardware = Patient P001

# ── Simulated vitals (replace with your real serial read function) ────────────
def get_simulated_vitals() -> dict:
    t = time.time()
    return {
        "hr_pulse":    int(60 + (t % 60)),
        "hr_max30100": int(72 + (t / 5) % 50),
        "spo2":        round(99 - (t / 10) % 9, 1),
        "body_temp":   round(36.5 + (t / 30) % 2, 1),
        "room_temp":   22.0,
        "humidity":    60.0
    }

def display_data(data: dict, timestamp: str) -> None:
    print(f"\n{'─' * 55}")
    print(f"  Timestamp    : {timestamp}")
    print(f"{'─' * 55}")
    print(f"  hr_pulse     : {data['hr_pulse']} BPM")
    print(f"  hr_max30100  : {data['hr_max30100']} BPM")
    print(f"  spo2         : {data['spo2']} %")
    print(f"  body_temp    : {data['body_temp']} °C")
    print(f"  room_temp    : {data['room_temp']} °C")
    print(f"  humidity     : {data['humidity']} %")
    if data['spo2'] < 94:
        print(f"  🔴 CRITICAL  : SpO2 low!")
    if data['hr_max30100'] > 110:
        print(f"  🔴 CRITICAL  : Tachycardia!")
    if data['body_temp'] > 38.3:
        print(f"  🟡 WARNING   : Fever!")
    print(f"{'─' * 55}")

def post_to_dashboard(data: dict) -> None:
    """Push vitals to the backend so dashboard updates in real-time"""
    try:
        payload = {"patient_id": PATIENT_ID, **data}
        resp = requests.post(BACKEND_URL, json=payload, timeout=2)
        if resp.status_code == 200:
            print(f"  ✅ Dashboard updated")
        else:
            print(f"  ⚠ Backend error: {resp.status_code}")
    except requests.exceptions.ConnectionError:
        print(f"  ⚠ Backend not reachable — is uvicorn running on port 8080?")
    except Exception as e:
        print(f"  ⚠ POST failed: {e}")

def log_to_file(data: dict, timestamp: str) -> None:
    record = {"timestamp": timestamp, "data": data}
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")

# ── Main Loop ─────────────────────────────────────────────────────────────────
def main():
    print("=" * 55)
    print("  Wokwi Real-Time Monitor")
    print(f"  Patient ID    : {PATIENT_ID}")
    print(f"  Dashboard URL : http://localhost:5500/index.html")
    print(f"  Backend URL   : {BACKEND_URL}")
    print("  Press Ctrl+C to stop.")
    print("=" * 55)

    while True:
        try:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            data      = get_simulated_vitals()   # 🔁 Replace with your serial read

            display_data(data, timestamp)
            post_to_dashboard(data)             # ← sends to dashboard

            if LOG_TO_FILE:
                log_to_file(data, timestamp)

            time.sleep(POLL_INTERVAL)

        except KeyboardInterrupt:
            print("\n\nMonitoring stopped. Goodbye!")
            break

main()

  Wokwi Real-Time Monitor
  Patient ID    : P001
  Dashboard URL : http://localhost:5500/index.html
  Backend URL   : http://localhost:8080/api/vitals
  Press Ctrl+C to stop.

───────────────────────────────────────────────────────
  Timestamp    : 2026-03-21 13:10:39.146
───────────────────────────────────────────────────────
  hr_pulse     : 99 BPM
  hr_max30100  : 89 BPM
  spo2         : 98.1 %
  body_temp    : 37.8 °C
  room_temp    : 22.0 °C
  humidity     : 60.0 %
───────────────────────────────────────────────────────
  ✅ Dashboard updated

───────────────────────────────────────────────────────
  Timestamp    : 2026-03-21 13:10:44.181
───────────────────────────────────────────────────────
  hr_pulse     : 104 BPM
  hr_max30100  : 90 BPM
  spo2         : 97.6 %
  body_temp    : 38.0 °C
  room_temp    : 22.0 °C
  humidity     : 60.0 %
───────────────────────────────────────────────────────
  ✅ Dashboard updated

───────────────────────────────────────────────────────
  Timestamp

In [13]:
import time
import json
from datetime import datetime

POLL_INTERVAL = 1.0
LOG_TO_FILE   = True
LOG_FILE      = "wokwi_log.json"

_start_time = time.time()

def get_vitals() -> dict:
    """
    3 phases — 5 minutes each:
    Phase 0 (0-5min)   → NORMAL
    Phase 1 (5-10min)  → WARNING
    Phase 2 (10-15min) → CRITICAL
    Then repeats
    """
    elapsed = time.time() - _start_time
    phase   = int(elapsed / 300) % 3  # 300 seconds = 5 minutes

    if phase == 0:
        vitals = {
            "hr_pulse":    72,
            "hr_max30100": 75,
            "spo2":        98.0,
            "body_temp":   36.6,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "NORMAL"
        }
    elif phase == 1:
        vitals = {
            "hr_pulse":    95,
            "hr_max30100": 98,
            "spo2":        95.0,
            "body_temp":   37.8,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "WARNING"
        }
    else:
        vitals = {
            "hr_pulse":    118,
            "hr_max30100": 122,
            "spo2":        91.0,
            "body_temp":   38.7,
            "room_temp":   22.0,
            "humidity":    60.0,
            "phase":       "CRITICAL"
        }

    icon = "✅" if phase == 0 else "🟡" if phase == 1 else "🚨"
    mins_remaining = int((300 - (elapsed % 300)) / 60)
    secs_remaining = int((300 - (elapsed % 300)) % 60)

    print(f"[VITALS] {icon} {vitals['phase']} | "
          f"HR={vitals['hr_max30100']} | "
          f"SpO2={vitals['spo2']}% | "
          f"Temp={vitals['body_temp']}°C | "
          f"Next phase in {mins_remaining}m {secs_remaining}s")
    return vitals


def display_data(data: dict, timestamp: str) -> None:
    phase = data.get("phase", "UNKNOWN")
    icon  = "✅" if phase == "NORMAL" else "🟡" if phase == "WARNING" else "🚨"

    elapsed = time.time() - _start_time
    mins_remaining = int((300 - (elapsed % 300)) / 60)
    secs_remaining = int((300 - (elapsed % 300)) % 60)

    print(f"\n{'─' * 60}")
    print(f"  {icon} Phase        : {phase}  "
          f"(changes in {mins_remaining}m {secs_remaining}s)")
    print(f"  Timestamp    : {timestamp}")
    print(f"{'─' * 60}")
    print(f"  hr_pulse     : {data['hr_pulse']} BPM")
    print(f"  hr_max30100  : {data['hr_max30100']} BPM")
    print(f"  spo2         : {data['spo2']} %")
    print(f"  body_temp    : {data['body_temp']} °C")
    print(f"  room_temp    : {data['room_temp']} °C")
    print(f"  humidity     : {data['humidity']} %")

    if data['spo2'] < 94:
        print(f"  🔴 CRITICAL  : SpO2 = {data['spo2']}% — Hypoxemia!")
    if data['hr_max30100'] > 110:
        print(f"  🔴 CRITICAL  : HR = {data['hr_max30100']} BPM — Tachycardia!")
    if data['body_temp'] > 38.3:
        print(f"  🟡 WARNING   : Temp = {data['body_temp']}°C — Fever!")
    if data['spo2'] < 94 and data['hr_max30100'] > 110:
        print(f"  🚨 RISK      : SEPSIS/PE — Escalate Immediately!")

    print(f"{'─' * 60}")


def log_to_file(data: dict, timestamp: str, filepath: str) -> None:
    record = {"timestamp": timestamp, "data": data}
    with open(filepath, "a") as f:
        f.write(json.dumps(record) + "\n")


def main() -> None:
    print("=" * 60)
    print("  VITAL Watch — Wokwi Real-Time Monitor")
    print("  Phases (5 minutes each):")
    print("  ✅  0-5min  → NORMAL   (HR=75  SpO2=98%  Temp=36.6°C)")
    print("  🟡  5-10min → WARNING  (HR=98  SpO2=95%  Temp=37.8°C)")
    print("  🚨 10-15min → CRITICAL (HR=122 SpO2=91%  Temp=38.7°C)")
    print("  Then repeats...")
    print(f"  Interval : {POLL_INTERVAL}s")
    print("  Press Ctrl+C to stop.")
    print("=" * 60)

    while True:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        data = get_vitals()
        display_data(data, timestamp)
        if LOG_TO_FILE:
            log_to_file(data, timestamp, LOG_FILE)
        try:
            time.sleep(POLL_INTERVAL)
        except KeyboardInterrupt:
            print("\n\nMonitoring stopped. Goodbye!")
            break


if __name__ == "__main__":
    main()

  VITAL Watch — Wokwi Real-Time Monitor
  Phases (5 minutes each):
  ✅  0-5min  → NORMAL   (HR=75  SpO2=98%  Temp=36.6°C)
  🟡  5-10min → WARNING  (HR=98  SpO2=95%  Temp=37.8°C)
  🚨 10-15min → CRITICAL (HR=122 SpO2=91%  Temp=38.7°C)
  Then repeats...
  Interval : 1.0s
  Press Ctrl+C to stop.
[VITALS] ✅ NORMAL | HR=75 | SpO2=98.0% | Temp=36.6°C | Next phase in 4m 59s

────────────────────────────────────────────────────────────
  ✅ Phase        : NORMAL  (changes in 4m 59s)
  Timestamp    : 2026-03-21 14:14:42.603
────────────────────────────────────────────────────────────
  hr_pulse     : 72 BPM
  hr_max30100  : 75 BPM
  spo2         : 98.0 %
  body_temp    : 36.6 °C
  room_temp    : 22.0 °C
  humidity     : 60.0 %
────────────────────────────────────────────────────────────
[VITALS] ✅ NORMAL | HR=75 | SpO2=98.0% | Temp=36.6°C | Next phase in 4m 58s

────────────────────────────────────────────────────────────
  ✅ Phase        : NORMAL  (changes in 4m 58s)
  Timestamp    : 2026-03-21 1